# Ensemble of methods

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abel-ai-lab/andrey/blob/main/examples/ensemble.ipynb)

Each method rests on assumptions, and you often do not know which of them hold for your data. An
ensemble fits several methods on the same data and keeps what they agree on. This notebook builds a
simple one from parts Andrey already has: four methods, a vote over their edges, and
`andrey.bootstrap` to weigh each vote by how stable it is. It then measures the ensemble against
the four methods on simulated data with a known graph and on the Sachs protein measurements. The
ensemble is a recipe to adapt, not one of Andrey's methods, and the measurements show where it
helps and where it does not.

## Start

Choose a CPU runtime and run this cell first. In Colab it installs `andrey-core==0.16.0` from PyPI;
local kernels use the installed environment.

In [1]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    os.environ["ANDREY_DEVICE"] = "cpu"
    !pip install --quiet "andrey-core[viz]==0.16.0" pandas

In [2]:
import warnings

import numpy as np
import pandas as pd
from IPython.display import HTML
from scipy import stats

import andrey
import andrey.viz
from andrey.core import ARROW, TAIL
from andrey.data import load_dataset, sample_scm
from andrey.metrics import shd, skeleton_scores, to_cpdag

## The data

We simulate 500 rows of 10 variables from a linear model on a random DAG with 15 edges: each
variable is a weighted sum of its parents plus noise, and the noise is uniform, so it is not
Gaussian. Because the data are simulated, we know the true graph, drawn here.
`andrey.viz.layout` fixes each node's place, so every drawing below puts the nodes where this one
does.

In [3]:
sample = sample_scm(functional="linear", noise="uniform", d=10, n=500, seed=1, density=3.0)
data, truth = sample.data, sample.graph
positions = andrey.viz.layout(truth)
andrey.viz.draw(truth, positions=positions)

'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 464 316" width="100%" font-family="Inter, system-ui, sans-serif"><rect width="464" height="316" fill="#fafbfc"/><path d="M 211.35 139.10 Q 116.08 185.83 40.47 273.94" fill="none" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="220.33,134.69 209.15,134.61 213.55,143.59" fill="#3a4658" stroke-linejoin="round"/><line x1="209.06" y1="127.30" x2="44.97" y2="115.39" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.03,128.03 209.42,122.32 208.70,132.29" fill="#3a4658" stroke-linejoin="round"/><line x1="211.30" y1="118.95" x2="143.70" y2="86.24" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="220.30,123.31 213.47,114.45 209.12,123.45" fill="#3a4658" stroke-linejoin="round"/><line x1="245.00" y1="128.97" x2="309.00" y2="128.97" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="319.00,128.97 309.00,123.97 309.00,133.97" fill="#3a4658" stroke-linejoin="round"/><line x1="244.34" y1="124.89" x2="410.16" y2="70.06" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="419.66,66.92 408.59,65.31 411.73,74.81" fill="#3a4658" stroke-linejoin="round"/><line x1="42.60" y1="276.28" x2="113.24" y2="226.15" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="121.40,220.36 110.35,222.07 116.14,230.23" fill="#3a4658" stroke-linejoin="round"/><line x1="45.00" y1="283.81" x2="109.00" y2="283.81" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="119.00,283.81 109.00,278.81 109.00,288.81" fill="#3a4658" stroke-linejoin="round"/><line x1="41.27" y1="123.57" x2="115.60" y2="196.71" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="122.73,203.72 119.11,193.14 112.10,200.27" fill="#3a4658" stroke-linejoin="round"/><path d="M 42.53 106.83 Q 122.11 49.28 209.27 35.73" fill="none" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.15,34.19 208.50,30.79 210.04,40.67" fill="#3a4658" stroke-linejoin="round"/><line x1="38.61" y1="125.65" x2="120.31" y2="264.00" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="125.39,272.61 124.61,261.46 116.00,266.54" fill="#3a4658" stroke-linejoin="round"/><line x1="44.31" y1="110.28" x2="110.22" y2="87.96" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="119.69,84.75 108.61,83.22 111.82,92.70" fill="#3a4658" stroke-linejoin="round"/><line x1="44.89" y1="112.79" x2="409.19" y2="65.78" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="419.11,64.50 408.55,60.82 409.83,70.74" fill="#3a4658" stroke-linejoin="round"/><line x1="209.00" y1="212.84" x2="145.00" y2="212.84" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.00,212.84 209.00,207.84 209.00,217.84" fill="#3a4658" stroke-linejoin="round"/><line x1="211.30" y1="42.21" x2="143.70" y2="74.92" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="220.30,37.86 209.12,37.71 213.47,46.71" fill="#3a4658" stroke-linejoin="round"/><line x1="342.84" y1="121.80" x2="412.82" y2="75.53" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="421.16,70.01 410.06,71.36 415.57,79.70" fill="#3a4658" stroke-linejoin="round"/><circle cx="232.00" cy="128.97" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="232.00" y="128.97" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">0</text><circle cx="32.00" cy="283.81" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="32.00" y="283.81" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">1</text><circle cx="32.00" cy="114.45" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="32.00" y="114.45" text-anchor="middle" font-size="11" font-weight="600" fill="

## Four methods on the same data

The ensemble's members should differ in how they work, so that they do not all make the same
mistakes. We take four:

- `pc` removes an edge when a test finds two variables independent;
- `ges` and `boss` search for the graph with the best score, `ges` over graphs and `boss` over
  orderings of the variables;
- `direct_lingam` reads the direction of each edge from noise that is not Gaussian.

All four assume linear relations and no hidden common causes. The first three return a CPDAG, where
an undirected edge (`--`) means that the data, as the method reads them, fit either direction.
DirectLiNGAM returns a DAG, with every edge directed. Its extra assumption holds here, since the
noise is uniform. PC's default test and the BIC score of GES and BOSS are built for Gaussian noise:
on these data they still aim at the right CPDAG, and they cannot use what the shape of the noise
says about direction.

We score each graph against the true DAG. `shd` counts the pairs of variables whose edge differs
from the truth, so a true edge left undirected is one error, as a missing, extra, or reversed edge
is. `skeleton_scores` looks only at which pairs are adjacent: precision is the share of a graph's
edges that are true, and recall the share of the true edges it has.

In [4]:
METHODS = ("pc", "ges", "boss", "direct_lingam")


def errors(graph, target):
    """SHD, and the precision and recall of the adjacencies, of `graph` against `target`."""
    adjacency = skeleton_scores(graph, target)
    return {
        "SHD": shd(graph, target),
        "adjacency precision": adjacency["precision"],
        "adjacency recall": adjacency["recall"],
    }


fits = {name: getattr(andrey, name)(data).structure for name in METHODS}
pd.DataFrame(
    [{"kind": graph.kind, **errors(graph, truth)} for graph in fits.values()], index=METHODS
).round(2)

,kind,SHD,adjacency precision,adjacency recall
pc,cpdag,11,1.00,0.67
ges,cpdag,9,0.81,0.87
boss,cpdag,5,0.93,0.93
direct_lingam,dag,0,1.00,1.00


DirectLiNGAM recovers the true graph on this dataset. PC adds no false edge and misses five of the
15 true ones, GES finds 13 of them and adds three false edges, and BOSS has one false edge and
misses one true edge. Here are the four graphs:

In [5]:
def side_by_side(graphs, positions):
    """Draw graphs two to a row, each under its name, with every node in the same place."""
    figures = "".join(
        f"<figure style='margin: 0'><figcaption>{name}</figcaption>"
        f"{andrey.viz.draw(graph, positions=positions)}</figure>"
        for name, graph in graphs.items()
    )
    grid = "display: grid; grid-template-columns: 1fr 1fr; gap: 16px"
    return HTML(f"<div style='{grid}'>{figures}</div>")


side_by_side(fits, positions)

## A common footing

A CPDAG and a DAG are different kinds of graph, so a vote needs one way to read both. Each graph
says one of four things about a pair of variables `a` and `b`: `a -> b`, `b -> a`, `a -- b`, or no
edge. Two arrays hold that: `edge[a, b]` is 1 when the pair has an edge, and `arrow[a, b]` is 1
when the graph says `a -> b`.

An undirected edge of a CPDAG says two things: the pair is adjacent, and the method cannot tell the
direction. So it counts as an edge, and as a vote for neither direction.

In [6]:
def claims(graph):
    """What `graph` says about each pair: `edge[a, b]` for an edge, `arrow[a, b]` for a -> b."""
    n = graph.n_nodes
    edge, arrow = np.zeros((n, n)), np.zeros((n, n))
    for a, b, kind in graph.oriented_edges(index=True):
        edge[a, b] = edge[b, a] = 1
        if kind == "directed":
            arrow[a, b] = 1
    return edge, arrow


def statement(edge, arrow, a, b):
    """The same reading in words, for the tables below."""
    if not edge[a, b]:
        return "no edge"
    return "->" if arrow[a, b] else "<-" if arrow[b, a] else "--"


def table_of_statements(graphs):
    """One row per pair that has an edge in some graph, one column per graph."""
    read = {name: claims(graph) for name, graph in graphs.items()}
    first = next(iter(graphs.values()))
    names = first.labels or range(first.n_nodes)
    return pd.DataFrame(
        {
            f"{names[a]}, {names[b]}": {name: statement(*read[name], a, b) for name in graphs}
            for a, b in zip(*np.triu_indices(first.n_nodes, 1))
            if any(edge[a, b] for edge, _ in read.values())
        }
    ).T


table_of_statements({**fits, "true graph": truth})

,pc,ges,boss,direct_lingam,true graph
"0, 1",<-,->,--,<-,<-
"0, 2",no edge,no edge,no edge,<-,<-
"0, 5",no edge,->,no edge,no edge,no edge
"0, 7",<-,->,->,<-,<-
"0, 8",->,--,--,->,->
"0, 9",no edge,->,->,->,->
"1, 2",no edge,<-,no edge,no edge,no edge
"1, 4",<-,->,->,->,->
"1, 6",<-,->,->,->,->
"1, 7",no edge,<-,->,no edge,no edge


Each row is a pair `a, b` with an edge in some graph: `->` reads `a -> b`, and `<-` reads
`b -> a`. The four methods agree on one row only. DirectLiNGAM alone has the true edge between 0
and 2, each of the three false edges (rows `0, 5`, `1, 2`, and `1, 7`) comes from one or two
methods, and on row `0, 1` the four graphs give three different readings.

## The vote

The rule has one setting, `k`:

- keep a pair when at least `k` methods have an edge between the two;
- direct a kept pair `a -> b` when more than half of the methods that have the edge say `a -> b`,
  and leave it undirected otherwise.

With four methods, `k=3` keeps the edges a majority has. The result holds directed and undirected
edges, so we label it `cpdag`, the kind that holds both. A vote is not always a valid CPDAG: its
directed edges need not be the ones a CPDAG fixes, and they can even close a cycle.

In [7]:
def combine(edge, arrow, k, labels=None):
    """The vote over methods; `edge` and `arrow` hold one array per method, as `claims` gives."""
    support = np.sum(edge, axis=0).round(6)  # how many methods have the pair
    forward = np.sum(arrow, axis=0).round(6)  # how many say a -> b
    keep = support >= k
    directed = keep & (forward > support / 2)
    marks = np.where(keep, TAIL, 0)  # a kept pair starts undirected: a tail at both ends
    marks[directed.T] = ARROW  # a -> b puts an arrowhead at b
    return andrey.GraphStructure.from_numpy(marks, kind="cpdag", labels=labels)


edge, arrow = zip(*(claims(graph) for graph in fits.values()))
vote = combine(edge, arrow, k=3)
print(errors(vote, truth))
side_by_side({"vote, k=3": vote, "true graph": truth}, positions)

{'SHD': 6, 'adjacency precision': 1.0, 'adjacency recall': 0.8666666666666667}


At `k=3` the vote keeps 13 edges, all of them true: none of the three false edges has three votes.
It loses two true edges, `0, 2` and `2, 7`, which fewer than three methods found, and it leaves
four true edges undirected, because no direction has more than half of their votes. That makes 6
errors: fewer than PC (11) and GES (9), one more than BOSS (5), and six more than DirectLiNGAM,
which has every edge right.

## Weigh each vote by stability

A single fit says yes or no to each edge, and a small change in the rows can flip a weak one.
`andrey.bootstrap` refits a method on resamples of the rows, here 20 subsamples of half of the rows
each, and returns the share of the fits that have each edge; the
[Bootstrap notebook](https://andrey.abel.ai/docs/examples/bootstrap.html) covers it. We replace
each method's yes or no by that share: `1 - frequency("absent")` is the share of the fits in which
a pair is adjacent, and `frequency("->")` the share in which it reads `a -> b`. The same `combine`
then adds shares instead of counting methods, so `k=3` asks that the four shares of a pair add up
to at least 3.

`bootstrap` also builds an averaged graph for the method, and warns when the fits have no valid
one. We read only the frequencies, so the cell silences that warning.

In [8]:
def stable_claims(method, data, seed, n_resamples=20):
    """`claims`, with the share of `n_resamples` refits of `method` in place of 0 or 1."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", andrey.AndreyWarning)
        out = andrey.bootstrap(method, data, n_resamples=n_resamples, seed=seed)
    return 1 - out.frequency("absent"), out.frequency("->")


shares, arrow_shares = zip(*(stable_claims(name, data, seed=1) for name in METHODS))
total = np.sum(shares, axis=0).round(6)
true_edge = claims(truth)[0]
pairs = [(a, b) for a, b in zip(*np.triu_indices(truth.n_nodes, 1)) if 1 <= total[a, b] < 3.9]
pd.DataFrame(
    [
        {
            **{name: share[a, b] for name, share in zip(METHODS, shares)},
            "total": total[a, b],
            "methods with the edge": int(sum(e[a, b] for e in edge)),
            "true edge": bool(true_edge[a, b]),
        }
        for a, b in pairs
    ],
    index=[f"{a}, {b}" for a, b in pairs],
)

,pc,ges,boss,direct_lingam,total,methods with the edge,true edge
"0, 2",0.00,0.45,0.45,1.00,1.90,1,True
"0, 7",0.60,0.85,1.00,1.00,3.45,4,True
"0, 9",0.45,1.00,1.00,1.00,3.45,3,True
"1, 7",0.00,0.80,0.60,0.05,1.45,2,False
"2, 6",0.05,1.00,1.00,1.00,3.05,3,True
"2, 7",0.35,0.30,0.90,1.00,2.55,2,True
"2, 9",0.85,1.00,1.00,1.00,3.85,4,True
"8, 9",0.00,1.00,0.95,1.00,2.95,3,True


The table lists the pairs whose shares add up to at least 1 and less than 3.9, the ones the fits
disagree on. A share says more than a yes or no. Three methods have the true edge between 8 and 9
in their fit on all the rows, but PC finds it in none of its subsamples, which hold half of the
rows, so the pair's total is 2.95. The false edge between 1 and 7, which GES and BOSS both return
on all the rows, is in 80% and 60% of their refits.

In [9]:
weighted = combine(shares, arrow_shares, k=3)
print(errors(weighted, truth))
side_by_side({"vote, k=3": vote, "weighted vote, k=3": weighted}, positions)

{'SHD': 4, 'adjacency precision': 1.0, 'adjacency recall': 0.8}


On this dataset the weighted vote at `k=3` has 4 errors where the plain vote has 6. It directs
three of the four edges that the plain vote leaves undirected, all three as the true graph does,
and it drops one more true edge, `8, 9`, whose shares add up to just under 3. A subsample of half
of the rows finds fewer weak edges than all the rows do, so here the same `k` asks more of a
weighted vote than of a plain one.

## Twenty datasets

One dataset shows how the recipe works, not how well. So we repeat it on 20 datasets, each a new
random graph and a new sample of 500 rows, and score every method, the vote at `k` of 2, 3, and 4,
and the weighted vote at `k` of 2 and 3. The table gives the mean of each score and, for SHD, a
95% interval for the mean (a t interval over the 20 datasets).

In [10]:
def graphs(data, seed):
    """Each method's graph on `data`, the votes over them, and the weighted votes."""
    result = {name: getattr(andrey, name)(data).structure for name in METHODS}
    labels = result["pc"].labels
    edge, arrow = zip(*(claims(graph) for graph in result.values()))
    for k in (2, 3, 4):
        result[f"vote, k={k}"] = combine(edge, arrow, k, labels)
    edge, arrow = zip(*(stable_claims(name, data, seed) for name in METHODS))
    for k in (2, 3):
        result[f"weighted vote, k={k}"] = combine(edge, arrow, k, labels)
    return result


def trials(noise, seeds=range(1, 21)):
    """The errors of every graph of `graphs` on one simulated dataset per seed."""
    rows = []
    for seed in seeds:
        sample = sample_scm(functional="linear", noise=noise, d=10, n=500, seed=seed, density=3.0)
        # Gaussian noise fixes the graph only up to its CPDAG, so that is the target there.
        target = to_cpdag(sample.graph) if noise == "gaussian" else sample.graph
        for name, graph in graphs(sample.data, seed).items():
            rows.append({"seed": seed, "graph": name, **errors(graph, target)})
    return pd.DataFrame(rows)


def interval(values):
    """The mean of `values` and the ends of its 95% t interval."""
    half = stats.t.ppf(0.975, len(values) - 1) * values.std(ddof=1) / np.sqrt(len(values))
    return values.mean(), values.mean() - half, values.mean() + half


def summary(results):
    """The mean of each score per graph, with the interval of the mean SHD."""
    table = results.groupby("graph", sort=False).mean().drop(columns="seed").rename_axis(None)
    ends = [interval(results.loc[results["graph"] == name, "SHD"]) for name in table.index]
    # An SHD is never below 0, so the interval stops there.
    table.insert(1, "95% interval", [f"{max(low, 0):.1f} to {high:.1f}" for _, low, high in ends])
    return table.round(2)


uniform = trials("uniform")
summary(uniform)

,SHD,95% interval,adjacency precision,adjacency recall
pc,11.20,9.9 to 12.5,0.95,0.67
ges,11.20,8.4 to 14.0,0.76,0.91
boss,5.70,3.8 to 7.6,0.95,0.97
direct_lingam,0.15,0.0 to 0.3,0.99,1.00
"vote, k=2",5.75,4.4 to 7.1,0.96,0.98
"vote, k=3",5.85,4.6 to 7.1,1.00,0.92
"vote, k=4",9.30,8.1 to 10.5,1.00,0.65
"weighted vote, k=2",3.55,2.5 to 4.6,0.99,0.95
"weighted vote, k=3",6.40,5.3 to 7.5,1.00,0.73


DirectLiNGAM, whose assumption holds, is nearly exact, with 0.15 errors on average, and no
ensemble comes close to it. Among the votes, a higher `k` trades recall for precision: at `k=2`
the vote has 98% of the true edges and 96% of its edges are true, and at `k=4` its precision is
1.00 and it has 65% of the true edges. The weighted vote makes the same trade at a lower `k`.

Is a difference between two rows more than noise? Every graph comes from the same 20 datasets, so
we take the difference in SHD dataset by dataset. Each cell below is the mean SHD of the ensemble
in its row minus that of the method in its column, with a 95% interval. A negative value means
that the ensemble has fewer errors, and an interval that holds 0 means that these 20 datasets do
not show a difference.

In [11]:
def differences(results):
    """Mean SHD of each ensemble minus each method's, dataset by dataset, with its interval."""
    by_seed = results.pivot(index="seed", columns="graph", values="SHD")
    ensembles = [name for name in by_seed.columns if name not in METHODS]
    return pd.DataFrame(
        {
            method: {
                name: "{:+.1f} ({:+.1f} to {:+.1f})".format(
                    *interval(by_seed[name] - by_seed[method])
                )
                for name in sorted(ensembles)
            }
            for method in METHODS
        }
    )


differences(uniform)

,pc,ges,boss,direct_lingam
"vote, k=2",-5.5 (-6.6 to -4.3),-5.5 (-8.1 to -2.8),+0.1 (-1.3 to +1.4),+5.6 (+4.3 to +6.9)
"vote, k=3",-5.3 (-6.4 to -4.3),-5.3 (-8.0 to -2.7),+0.1 (-1.0 to +1.3),+5.7 (+4.5 to +6.9)
"vote, k=4",-1.9 (-2.6 to -1.2),-1.9 (-4.3 to +0.5),+3.6 (+2.2 to +5.0),+9.2 (+8.0 to +10.3)
"weighted vote, k=2",-7.7 (-9.2 to -6.1),-7.7 (-10.6 to -4.7),-2.1 (-4.1 to -0.2),+3.4 (+2.4 to +4.4)
"weighted vote, k=3",-4.8 (-5.8 to -3.8),-4.8 (-7.2 to -2.4),+0.7 (-1.1 to +2.5),+6.2 (+5.2 to +7.3)


Every ensemble has more errors than DirectLiNGAM, 3.4 to 9.2 more on average, and every one of
those intervals is above 0: here the ensemble is worse than its best member. Every ensemble has
fewer errors than PC, and all but the vote at `k=4` have fewer than GES. Against BOSS, the votes
at `k` of 2 and 3 and the weighted vote at `k=3` show no difference, the vote at `k=4` is worse,
and the weighted vote at `k=2` has 2.1 fewer errors, with an interval that ends just below 0. A
table of 20 intervals can hold one that misses 0 by chance, so that cell is weak evidence.

## When the noise is Gaussian

Now change one thing: the noise is Gaussian. DirectLiNGAM's extra assumption fails, and no method
can tell the DAGs of one equivalence class apart, so the target is the true CPDAG. The other three
methods' assumptions hold exactly.

In [12]:
gaussian = trials("gaussian")
summary(gaussian)

,SHD,95% interval,adjacency precision,adjacency recall
pc,9.90,8.7 to 11.1,0.92,0.68
ges,9.05,6.0 to 12.1,0.82,0.92
boss,4.65,2.1 to 7.2,0.93,0.95
direct_lingam,20.05,17.5 to 22.6,0.57,0.95
"vote, k=2",8.50,6.1 to 10.9,0.82,0.96
"vote, k=3",6.05,4.3 to 7.8,0.95,0.90
"vote, k=4",8.25,7.0 to 9.5,1.00,0.65
"weighted vote, k=2",6.15,4.7 to 7.6,0.91,0.93
"weighted vote, k=3",7.40,6.3 to 8.5,1.00,0.68


In [13]:
differences(gaussian)

,pc,ges,boss,direct_lingam
"vote, k=2",-1.4 (-3.7 to +0.9),-0.6 (-2.8 to +1.7),+3.9 (+1.6 to +6.1),-11.6 (-14.3 to -8.8)
"vote, k=3",-3.9 (-5.6 to -2.1),-3.0 (-5.8 to -0.2),+1.4 (-0.4 to +3.2),-14.0 (-16.8 to -11.2)
"vote, k=4",-1.6 (-2.6 to -0.7),-0.8 (-3.8 to +2.2),+3.6 (+1.6 to +5.6),-11.8 (-14.6 to -9.0)
"weighted vote, k=2",-3.8 (-5.5 to -2.0),-2.9 (-5.1 to -0.7),+1.5 (-1.2 to +4.2),-13.9 (-16.4 to -11.4)
"weighted vote, k=3",-2.5 (-3.8 to -1.2),-1.6 (-4.2 to +0.9),+2.8 (+0.7 to +4.8),-12.7 (-15.1 to -10.2)


The order of the methods turns over. DirectLiNGAM, the best method under uniform noise, is now the
worst, with 20 errors on average and a precision of 0.57: without its assumption, its directions
and many of its edges are wrong. BOSS is the best, with 4.65 errors. Every ensemble has 12 to 14
fewer errors than DirectLiNGAM, and none has fewer than BOSS: the vote at `k=3` and the weighted
vote at `k=2` show no difference from it, and the other three are worse.

Across the two settings, DirectLiNGAM has 0.15 and 20.05 errors, the vote at `k=3` has 5.85 and
6.05, and BOSS has 5.70 and 4.65. The vote is never the best and never the worst, without being
told which setting it is in. BOSS, one method, does as well as that vote in both.

## Sachs

The Sachs data hold 11 proteins and phospholipids measured in 853 human immune cells, with a
reference network of 17 edges that biologists had mapped; the
[Quick start notebook](https://andrey.abel.ai/docs/examples/quick_start.html) introduces them.
The reference is the literature's consensus, not a known truth, and it is one dataset, so the
scores have no interval: a difference of a few edges here is not evidence.

In [14]:
sachs = load_dataset("sachs")
on_sachs = graphs(sachs.data, seed=1)
print("the vote at k=3 is PC's graph:", on_sachs["vote, k=3"] == on_sachs["pc"])
pd.DataFrame(
    [
        {"edges": len(graph.oriented_edges()), **errors(graph, sachs.graph)}
        for graph in on_sachs.values()
    ],
    index=list(on_sachs),
).round(2)

the vote at k=3 is PC's graph: True


,edges,SHD,adjacency precision,adjacency recall
pc,8,17,1.00,0.47
ges,8,17,1.00,0.47
boss,8,17,1.00,0.47
direct_lingam,11,14,0.82,0.53
"vote, k=2",8,17,1.00,0.47
"vote, k=3",8,17,1.00,0.47
"vote, k=4",8,17,1.00,0.47
"weighted vote, k=2",7,17,1.00,0.41
"weighted vote, k=3",7,17,1.00,0.41


PC, GES, and BOSS return the same eight edges, and DirectLiNGAM returns 11. Every plain vote is
that graph of eight edges again, since DirectLiNGAM has an edge on each of the eight pairs too,
and the weighted votes keep seven of them. All five score 17 errors, which is what a graph with no
edges scores against a reference of 17 edges. DirectLiNGAM scores 14, and one dataset cannot tell
14 from 17. Here is what each graph says about each pair, and the vote beside the reference:

In [15]:
table_of_statements({**{name: on_sachs[name] for name in METHODS}, "reference": sachs.graph})

,pc,ges,boss,direct_lingam,reference
"raf, mek",--,--,--,<-,->
"raf, pka",no edge,no edge,no edge,no edge,<-
"raf, pkc",no edge,no edge,no edge,no edge,<-
"mek, pip2",no edge,no edge,no edge,->,no edge
"mek, erk",no edge,no edge,no edge,no edge,->
"mek, pka",no edge,no edge,no edge,no edge,<-
"mek, pkc",no edge,no edge,no edge,no edge,<-
"plc, pip2",no edge,no edge,no edge,->,->
"plc, pip3",--,--,--,->,->
"pip2, pip3",--,--,--,->,<-


In [16]:
side_by_side(
    {"vote, k=3": on_sachs["vote, k=3"], "reference network": sachs.graph},
    andrey.viz.layout(sachs.graph),
)

The eight pairs that PC, GES, and BOSS agree on are all adjacent in the reference, so the vote's
precision is 1.00. Six of them stay undirected, and the two that are directed, both into pkc, point
against the reference. Eight reference edges are in none of the four graphs, and a ninth, plc ->
pip2, is in DirectLiNGAM's only. The agreement of the three methods says little here, because they
agree for a shared reason: all three work from the correlations between the variables, under the
same assumptions, and what those miss, all three miss.

## What an ensemble can and cannot do

The measurements above say five things.

- **Agreement trades recall for precision.** A higher `k` removes false edges and true ones. In
  both simulated settings the vote's precision rose from `k=2` to `k=4`, from 0.96 and 0.82 to
  1.00, and its recall fell, from 0.98 and 0.96 to 0.65. Choose `k` by which error costs you more.
- **A vote cannot use what only one method knows.** With uniform noise only DirectLiNGAM could
  read the directions, and every ensemble had 3.4 to 9.2 more errors than it did.
- **Methods that share an assumption fail together.** On Sachs, three methods returned the same
  graph, the vote returned it again, and it scored no better than a graph with no edges.
  Agreement among methods that read the data the same way is not evidence that they are right.
- **An ensemble is a hedge, not an improvement.** The best method changed with the noise, and no
  ensemble beat the best method of either setting. The vote at `k=3` stayed near 6 errors in both,
  where DirectLiNGAM went from 0.15 to 20, and BOSS alone did as well as that vote. When you know
  which assumptions your data meet, use the method built on them: each method's documentation
  says when to use it.
- **A share is not a probability.** The share of the refits with an edge says how stable the edge
  is, which ranks the edges; it does not say how likely the edge is to be true. It also depends
  on the resampling: with half of the rows, weak edges get lower shares, so at `k=3` the weighted
  vote had a recall of 0.73 under uniform noise where the plain vote had 0.92.

## To adapt the recipe

Change the members, `k`, or the reading of an undirected edge. Members that share an assumption
share its failures, so choose methods that differ. Counting `--` as no vote on direction, so that
a pair is directed when more methods point it one way than the other, lets one method direct what
the others leave open, and that direction then rests on its assumption alone.
`andrey.BootstrapOutput.from_fits` counts any list of fits as `bootstrap` counts its refits, and
makes the vote a valid graph.

The published work on combining graphs goes further than this recipe. Friedman, Goldszmidt, and
Wyner (1999) used the bootstrap to measure confidence in the features of a learned network, such
as its edges. Meinshausen and Buhlmann (2010) keep what is selected in most subsamples and, under
conditions they state, bound the expected number of false selections as a function of the
threshold. Scutari and Nagarajan (2013) estimate the threshold on edge frequencies from the
frequencies themselves, in place of a fixed value.

## References

- Friedman, N., Goldszmidt, M., and Wyner, A. (1999). Data analysis with Bayesian networks: a
  bootstrap approach. Proceedings of the Fifteenth Conference on Uncertainty in Artificial
  Intelligence, 196-205.
- Meinshausen, N., and Buhlmann, P. (2010). Stability selection. Journal of the Royal Statistical
  Society: Series B, 72(4), 417-473.
- Scutari, M., and Nagarajan, R. (2013). Identifying significant edges in graphical models of
  molecular networks. Artificial Intelligence in Medicine, 57(3), 207-217.